In [25]:
import networkx as nx
import numpy as np
import pandas as pd

In [5]:
cluster_graph = "/mnt/home/ssledzieski/Projects/philharmonic-experiments/results/p_damicornis/20240917-pdamAll-phil_v0.3.0/20240917-pdamAll-phil_v0.3.0_cluster_graph.tsv"
cluster_graph_functions = "/mnt/home/ssledzieski/Projects/philharmonic-experiments/results/p_damicornis/20240917-pdamAll-phil_v0.3.0/20240917-pdamAll-phil_v0.3.0_cluster_graph_functions.tsv"

df = pd.read_csv(cluster_graph, sep="\t")
G = nx.from_pandas_edgelist(df, "source", "target", edge_attr="weight")
df_functions = pd.read_csv(cluster_graph_functions, sep="\t")

In [ ]:
# Filter graph to >=50 edge weight

edge_weight_thresh = 50
filtered_edges = [
    (u, v) for u, v, d in G.edges(data=True) if d.get("weight", 0) >= edge_weight_thresh
]
filtered_G = G.edge_subgraph(filtered_edges).copy()

print("Original Nodes:", len(G.nodes()))
print("Filtered Nodes:", len(filtered_G.nodes()))
print("Filtered Edges:", len(filtered_G.edges(data=True)))

Original Nodes: 275
Filtered Nodes: 164
Filtered Edges: 838


In [10]:
# Compute shortest path distances between nodes
df["inv_weight"] = 1 / df["weight"]
closeness_G = nx.from_pandas_edgelist(df, "source", "target", edge_attr="inv_weight")

# dict-of-dicts: dist[u][v] = shortest path length from u to v
dist = dict(nx.all_pairs_dijkstra_path_length(closeness_G, weight="inv_weight"))
nodes = list(G.nodes())
dist_df = pd.DataFrame(dist, index=nodes, columns=nodes)

In [48]:
def weighted_attr_assortativity(G, attr, weight="weight"):
    cats = sorted({G.nodes[n][attr] for n in G.nodes()}, key=str)
    idx = {c: i for i, c in enumerate(cats)}
    M = np.zeros((len(cats), len(cats)))
    for u, v, d in G.edges(data=True):
        w = d.get(weight, 1.0)
        i, j = idx[G.nodes[u][attr]], idx[G.nodes[v][attr]]
        M[i, j] += w
        if i != j:
            M[j, i] += w
    M /= M.sum()
    a = M.sum(axis=1)
    return (np.trace(M) - (a * a).sum()) / (1 - (a * a).sum())


def analyze_label_clustering(G, df_functions, top_k=None, name=""):
    # Attach all labels (not just top-k) so assortativity sees full label structure
    labels = df_functions.set_index("key")["go_fn"].dropna()
    labels = labels[labels.index.isin(G.nodes())]
    if top_k is None:
        top_k = len(labels.unique())

    H = G.copy()
    nx.set_node_attributes(H, labels.to_dict(), name="label")
    H.remove_nodes_from([n for n in H.nodes() if "label" not in H.nodes[n]])

    # Identify top-k labels
    label_counts = pd.Series([H.nodes[n]["label"] for n in H.nodes()]).value_counts()
    top_labels = set(label_counts.head(top_k).index)
    top_nodes = [n for n in H.nodes() if H.nodes[n]["label"] in top_labels]

    print(f"\n=== {name} (n={H.number_of_nodes()}, m={H.number_of_edges()}) ===")
    print(f"Top {top_k} labels: {sorted(top_labels, key=str)}")
    print(
        f"Assortativity over all labels (unweighted): {nx.attribute_assortativity_coefficient(H, 'label'):.4f}"
    )
    print(
        f"Assortativity over all labels (weighted):   {weighted_attr_assortativity(H, 'label'):.4f}"
    )

    # ---- Conductance ----
    label_to_nodes = {}
    for n in top_nodes:
        label_to_nodes.setdefault(H.nodes[n]["label"], []).append(n)
    cond_df = pd.DataFrame(
        [
            {
                "label": lab,
                "n_clusters": len(members),
                "cond_weighted": nx.conductance(H, members, weight="weight"),
                "cond_unweighted": nx.conductance(H, members, weight=None),
            }
            for lab, members in label_to_nodes.items()
        ]
    )

    print("\nPer-label summary (top-k only, distances on full graph):")
    print(cond_df.sort_values(by="n_clusters", ascending=False).to_string(index=False))
    return cond_df

In [50]:
full_summary = analyze_label_clustering(G, df_functions, name="full graph")
filt_summary = analyze_label_clustering(
    filtered_G, df_functions, name="filtered (w>=50)"
)


=== full graph (n=173, m=1460) ===
Top 18 labels: ['DNA recombination', 'DNA replication', 'DNA-templated transcription', 'carbohydrate metabolic process', 'cell adhesion', 'cytoplasmic translation', 'cytoskeleton organization', 'extracellular matrix organization', 'inflammatory response', 'intracellular protein transport', 'microtubule-based movement', 'mitotic cell cycle', 'photosynthesis', 'protein glycosylation', 'regulation of DNA-templated transcription', 'respiratory system process', 'ribosome biogenesis', 'wound healing']
Assortativity over all labels (unweighted): 0.2376
Assortativity over all labels (weighted):   0.2959

Per-label summary (top-k only, distances on full graph):
                                    label  n_clusters  cond_weighted  cond_unweighted
                       mitotic cell cycle          48       0.475573         0.594839
                    inflammatory response          41       0.101809         0.323263
              DNA-templated transcription    